In [4]:
# Reddit Pushshift Filter
# Filters monthly dump files for K-pop comeback related posts
# K-pop Sentiment Analysis Project

import zstandard as zstd
import json
import pandas as pd
import os
from datetime import datetime, timezone

# --- Comeback windows (14 days around each release date) ---
COMEBACKS = {
    "StrayKids_ChkChkBoom": {
        "start": "2024-07-12",
        "end": "2024-07-26",
        "keywords": ["stray kids", "straykids", "skz", "chk chk boom", "chkchkboom", "ate album"]
    },
    "SEVENTEEN_Thunder": {
        "start": "2024-10-07",
        "end": "2024-10-21",
        "keywords": ["seventeen", "carats", "spill the feels", "thunder seventeen", "17 thunder"]
    },
    "aespa_Whiplash": {
        "start": "2024-10-14",
        "end": "2024-10-28",
        "keywords": ["aespa", "whiplash aespa", "aespa whiplash", "næspa"]
    },
    "NCTDREAM_WhenImWithYou": {
        "start": "2024-11-04",
        "end": "2024-11-18",
        "keywords": ["nct dream", "nctdream", "dreamscape", "when im with you", "when i'm with you"]
    },
    "TWICE_Strategy": {
        "start": "2024-11-29",
        "end": "2024-12-13",
        "keywords": ["twice", "strategy twice", "twice strategy", "once fandom"]
    },
    "IVE_RebelHeart": {
        "start": "2025-01-27",
        "end": "2025-02-10",
        "keywords": ["ive kpop", "ive rebel heart", "rebel heart ive", "ive empathy", "dive fandom"]
    }
}

# --- Subreddits to include ---
TARGET_SUBREDDITS = [
    "kpop", "kpopthoughts", "unpopularkpopopinions",
    "straykids", "seventeen", "aespa", "nctdream", "twice", "ive"
]

def parse_date(date_str):
    return datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)

def is_relevant(post, comeback_name, comeback_info):
    # Check date
    created = datetime.fromtimestamp(post.get("created_utc", 0), tz=timezone.utc)
    start = parse_date(comeback_info["start"])
    end = parse_date(comeback_info["end"])
    
    if not (start <= created <= end):
        return False
    
    # Check subreddit
    subreddit = post.get("subreddit", "").lower()
    if subreddit not in TARGET_SUBREDDITS:
        return False
    
    # Check keywords in title or body
    text = (post.get("title", "") + " " + post.get("body", "") + " " + post.get("selftext", "")).lower()
    return any(keyword in text for keyword in comeback_info["keywords"])

def filter_dump(filepath, output_folder):
    print(f"\nProcessing: {filepath}")
    results = {name: [] for name in COMEBACKS}
    
    with open(filepath, "rb") as f:
        dctx = zstd.ZstdDecompressor(max_window_size=2**31)
        
        with dctx.stream_reader(f) as reader:
            buffer = ""
            chunk_size = 2**24  # 16MB chunks
            
            while True:
                chunk = reader.read(chunk_size)
                if not chunk:
                    break
                
                buffer += chunk.decode("utf-8", errors="ignore")
                lines = buffer.split("\n")
                buffer = lines[-1]
                
                for line in lines[:-1]:
                    if not line.strip():
                        continue
                    try:
                        post = json.loads(line)
                        for comeback_name, comeback_info in COMEBACKS.items():
                            if is_relevant(post, comeback_name, comeback_info):
                                results[comeback_name].append({
                                    "id": post.get("id"),
                                    "subreddit": post.get("subreddit"),
                                    "text": (post.get("title", "") + " " + post.get("body", "") + post.get("selftext", "")).strip(),
                                    "score": post.get("score", 0),
                                    "created_utc": post.get("created_utc"),
                                    "comeback": comeback_name
                                })
                    except json.JSONDecodeError:
                        continue
    
    # Save results
    for comeback_name, posts in results.items():
        if posts:
            df = pd.DataFrame(posts)
            out_path = os.path.join(output_folder, f"{comeback_name}_reddit_raw.csv")
            
            if os.path.exists(out_path):
                df.to_csv(out_path, mode="a", header=False, index=False)
            else:
                df.to_csv(out_path, index=False)
            
            print(f"  {comeback_name}: {len(posts)} posts saved")

# --- Run the filter ---
dump_folder = "../01_raw_data/reddit"
output_folder = "../01_raw_data/reddit"

# Find all .zst files in the reddit folder
dump_files = [f for f in os.listdir(dump_folder) if f.endswith(".zst")]
print(f"Found {len(dump_files)} dump files: {dump_files}")

for dump_file in dump_files:
    filepath = os.path.join(dump_folder, dump_file)
    filter_dump(filepath, output_folder)

print("\nAll done!")

Found 0 dump files: []

All done!
